In [1]:
import pandas as pd
import numpy as np
import joblib

import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

from keras.models import Sequential
from keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization, GlobalAveragePooling1D
from keras.optimizers import Adam

2025-08-26 10:32:12.796210: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


# Data Load and Claeaning

In [2]:
file_path = "low_freq_data/low_freq_all_data.csv"  
df = pd.read_csv(file_path)
print("Original shape:", df.shape)

Original shape: (3674, 506)


/tmp/ipykernel_615318/137253656.py:2: DtypeWarning: Columns (503) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


In [3]:
X = df[[f'f{i}' for i in range(500)]].to_numpy(dtype=float)
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [4]:
df_test = pd.read_csv("low_freq_data/low_freq_test_normal_day.csv")
print("Original shape:", df_test.shape)

Original shape: (281045, 506)


In [5]:
locs = ['[4220]', '[4230]', '[4240]', '[4250]']
more_train = df_test[df_test['location'].isin(locs)]

more_X = more_train[[f'f{i}' for i in range(500)]].to_numpy(dtype=float)
more_y = more_train['label']
more_y = le.transform(more_y)

In [6]:
X_train = np.concatenate([X_train, more_X], axis=0)
y_train = np.concatenate([y_train, more_y], axis=0)

In [7]:
X_train.shape

(5335, 500)

# XGBoost

In [ ]:
xgb = XGBClassifier()

param_grid_xgb = {
    'n_estimators': [100, 150, 200],
    'max_depth': [5, 7, 9],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

grid_search_xgb = GridSearchCV(estimator=xgb,
                           param_grid=param_grid_xgb,
                           cv=5,
                           scoring='accuracy',
                           verbose=1,
                           n_jobs=-1)


In [ ]:
grid_search_xgb.fit(X_train, y_train)

best_xgb = grid_search_xgb.best_estimator_

In [9]:
y_pred = best_xgb.predict(X_test)
# print("Best Parameters:", grid_search_xgb.best_params_)
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       359
           1       1.00      1.00      1.00       376

    accuracy                           1.00       735
   macro avg       1.00      1.00      1.00       735
weighted avg       1.00      1.00      1.00       735



In [ ]:
import joblib

# Save
joblib.dump(best_xgb, "xgb_classifier.pkl")

In [8]:
import pickle

with open("xgb_classifier.pkl", "rb") as f:
    best_xgb = pickle.load(f)

In [11]:
unique_locations = df_test['location'].unique()

for loc_str in unique_locations:

    if loc_str in locs:
        continue

    subset = df_test[df_test['location'] == loc_str]

    if subset.empty:
        continue

    X_t = subset[[f'f{i}' for i in range(500)]].to_numpy(dtype=float)
    y_t = subset['label']
    y_t = le.transform(y_t)

    y_pred =  best_xgb.predict(X_t)
    acc = accuracy_score(y_t, y_pred)

    if acc < 0.97:
        print(f"Location {loc_str}: Accuracy = {acc:.2%} ({len(subset)} samples)")

Location [3200]: Accuracy = 35.23% (599 samples)
Location [3440]: Accuracy = 81.47% (599 samples)
Location [3450]: Accuracy = 95.49% (599 samples)
Location [3460]: Accuracy = 87.31% (599 samples)
